<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/Titans_jax_sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 0. Environment Setup

# Clone the gemma repository
!git clone --depth 1  https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository for UI/UX
!git clone --depth 1  https://github.com/google-deepmind/dialog.git || true
!pip install -q ./dialog

# HuggingFace auth dependencies
!pip install -q python-dotenv huggingface_hub

# Ensure local modules are in path
import sys
import os
sys.path.append(os.getcwd())

In [ ]:
!git clone --depth 1  https://github.com/andrew-veriga/Titans_jax.git


In [ ]:
!pip install -q --upgrade --force-reinstall "tensorflow>=2.16"


In [ ]:
import importlib
import jax
import os

%cd Titans_jax

import gemma_titans
# importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils
from gemma import gm

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"

# jax.config.update("jax_disable_jit", True) # Temporarily disable JIT to bypass the hashing error

In [ ]:
import sys
import tensorflow as tf
print(sys.version)
print(tf.__version__)

In [ ]:
import dataclasses
import optax
titans_first_layer = 23  # Titans layers from this index onward are active.
                                 # Earlier layers revert to standard Gemma blocks.
                                 # 17 → layers 17,23 active (~25GB compile RAM)
                                 # 23 → layer 23 only  (~5GB compile RAM)

_all_titans_layers = (11, 17, 23)
active_titans_layers = tuple(l for l in _all_titans_layers if l >= titans_first_layer)
print(f"Active Titans layers: {active_titans_layers}")



In [ ]:
# HuggingFace authentication
# Token is read exclusively from the HF_TOKEN environment variable.
# Sources that populate it (auto-detected):
#   • local:  .env file via python-dotenv
#   • Colab:  Secrets (left sidebar 🔑) → exported to os.environ below
#   • shell:  export HF_TOKEN=...
import os

# Colab: copy Secret into os.environ so the code below is portable.
# (No-op outside Colab — ImportError is silently ignored.)
try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
except (ImportError, Exception):
    pass

# Single contract — always consume from the environment variable
from huggingface_hub import login

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    raise RuntimeError(
        'HF_TOKEN not found. Set the HF_TOKEN environment variable '
        '(export HF_TOKEN=..., Colab Secrets, or .env file).'
    )

login(HF_TOKEN)

In [ ]:
from gemma import gm
import numpy as np
import jax.numpy as jnp
import os
import orbax.checkpoint as ocp
HF_CKPT_REPO = "veriga/titans-checkpoints"

In [ ]:
import hf_checkpoint
importlib.reload(hf_checkpoint)
from hf_checkpoint import (
    save_checkpoint_to_hf, load_checkpoint_from_hf,
    save_last_metadata, load_last_metadata,
    reconstruct_opt_params, schedule,
    load_all_phase1_layers,
)

In [ ]:
def load_titans_weights(load_dir: str):
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(os.path.abspath(load_dir))

merged_params = None
workdir = os.path.abspath(f'./titans_workdir_phase2_from{titans_first_layer}')
workdir_checkpoints = os.path.join(workdir, "checkpoints")

if os.path.exists(workdir_checkpoints) and len(os.listdir(workdir_checkpoints)) > 0:
    print(f"📁 Найдена директория {workdir_checkpoints}. Пропускаем загрузку весов.")
    print("Kauldron автоматически загрузит последнее состояние при старте обучения.")
else:
    # Определяем, какую фазу загружать
    load_phase = 1
    phase_label = f"Phase {load_phase}"

    # ── Авто-определение последнего чекпойнта ──
    last_meta = load_last_metadata(
        repo_id=HF_CKPT_REPO,
        phase=load_phase
    )

    if last_meta is not None:
        # Восстанавливаем experimental_config
        experimental_config = last_meta.get("experimental_config", {})
        print(f"📋 Restored experimental_config: {experimental_config}")

        # Восстанавливаем opt_params: schedules → callable
        if "opt_params" in last_meta:
            opt_params = reconstruct_opt_params(last_meta["opt_params"])
            print(f"📋 Restored opt_params with schedules: {list(opt_params.keys())}")



    # ── Загружаем веса для ВСЕХ обученных слоёв из Phase 1 ──
    loaded_titans_params = load_all_phase1_layers(
        repo_id=HF_CKPT_REPO,
        titans_first_layer=titans_first_layer,
        local_dir=".",
    )

    if loaded_titans_params is not None:
        # Keep only weights for active Titans layers
        active_layer_keys = {f'layer_{l}' for l in active_titans_layers}
        loaded_titans_params = {
            k: v for k, v in loaded_titans_params.items()
            if k in active_layer_keys
        }
        print(f"Merging Titans weights for: {sorted(loaded_titans_params.keys())}")

        print("Loading Gemma base weights...")
        original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

        merged_params = titans_tree_utils.merge_titans_params(
            original_params, loaded_titans_params, remove_dead_attn=True
        )
        print(f"✅ Phase 1 weights loaded from HF and merged.")
    else:
        print("⚠️ Не найдено обученных слоёв Phase 1 на HF!")



print(f"avg gate bias: {np.mean((merged_params['layer_23']['memory_gate_proj']['bias']))}")
tokenizer = gm.text.Gemma3Tokenizer()

## 4. Interactive Dialogue with `google-deepmind/dialog`

In [ ]:
!git pull

In [ ]:
# import gemma_titans
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans

In [ ]:
neural_mem_kwargs = {
    **experimental_config,
    'every_k_schedule': opt_params['every_k_schedule'],
    'huber_loss_delta': experimental_config.get('huber_loss_delta', 0.5),  # явной фикс, если нужно
}
inference_config = dataclasses.replace(
    Gemma3_1B_Titans.config,
    is_training_mode=False,
    # sliding_window_size=128,
    titans_layer_indices=active_titans_layers,
    titans_first_layer=titans_first_layer,
    neural_mem_kwargs = neural_mem_kwargs

)
model = Gemma3_1B_Titans(
    config=inference_config,
    dtype=jnp.float32,
    return_last_only=False,
    tokens="batch.tokens",
)


In [ ]:
_TARGET_DTYPE = jnp.float32
merged_params = jax.tree_util.tree_map(lambda x: x.astype(_TARGET_DTYPE), merged_params)

In [ ]:
import dialog
from gemma import gm
import jax

# Initialize Sampler and Conversation
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

conv = dialog.Conversation()


def chat(user_input: str):
    global conv
    # Add user message
    conv += dialog.User(user_input)

    # Convert conversation to prompt (Gemma 3 format)
    prompt = conv.as_text(training=False)

    # Generate response
    # Note: Sampler handles the caching of Titans memory automatically
    response_text = sampler.sample(prompt, max_new_tokens=128)

    # Add model response to UI
    conv += dialog.Model(response_text)
    conv.show()

# Example usage:
# chat("Привет! Кто такие Титаны в мифологии?")

In [ ]:
chat("Who is Polyphemus in the ancient greek mythology?")
